# api_call — full client walkthrough

Talks to a running `dorna_vision` server and exercises every major call: lifecycle, detection, proxy RPC, camera control, robot, image fetch, retval inspection, and cleanup.

## 1. Connect

In [1]:
from dorna_vision_client import VisionClient

vc = VisionClient()
vc.connect()             # defaults: host="127.0.0.1", port=8765
vc.hello()

{'id': 2,
 'stat': 2,
 'server': 'dorna_vision',
 'version': '2.4.0',
 'protocol': 1}

## 2. Discover and add a camera

In [3]:
devs = vc.camera_list()
print("attached devices:", devs)

serial_number = devs[0]["serial_number"]
vc.camera_add(
    serial_number=serial_number,
    mode="bgrd",
    stream={"width": 848, "height": 480, "fps": 15},
)

attached devices: [{'name': 'Intel RealSense D405', 'serial_number': '130322274110', 'usb_type': '3.2', 'usb_port': '\\\\?\\usb#vid_8086&pid_0b5b&mi_00#6&2d556483&1&0000#{e5323777-f976-4f5b-9b55-b94699c46e44}\\global'}]


{'id': 5, 'stat': 2, 'serial_number': '130322274110'}

## 3. Robots — addressed by host

Robots don't need explicit registration — just reference them by IP. The first call to `vc.robot(host).X()` (or a `detection_add(robot_host=...)`) lazy-connects with default port 443 and model `dorna_ta`. For non-default port or model, call `vc.robot_add(host=..., port=..., model=...)` once to pre-connect.

In [ ]:
# Optional: pre-connect a robot with custom params.
# vc.robot_add(host="192.168.1.50", port=443, model="dorna_ta")

# Or just use it directly — lazy-connects on first call:
# vc.robot("192.168.1.50").joint()

## 4. Add two detections on the same camera

Detections are scoped to this client session. The same camera is shared.

In [5]:
vc.detection_add(
    name="cnt1",
    camera_serial_number=serial_number,
    detection={
        "cmd": "cnt",
        "type": 0, "inv": True, "blur": 3, "thr": 127,
        "circularity_min": 0, "convexity_min": 0,
    },
    limit={"bb": {"area": [500, 500000], "aspect_ratio": [], "inv": 0}},
    display={"label": 1},
)

vc.detection_add(
    name="blob1",
    camera_serial_number=serial_number,
    detection={"cmd": "blob"},
    display={"label": 1},
)

print("detections in this session:", vc.detection_list())

detections in this session: [{'name': 'cnt1', 'camera_serial_number': '130322274110', 'cmd': 'cnt'}, {'name': 'blob1', 'camera_serial_number': '130322274110', 'cmd': 'blob'}]


## 5. Run the detections

Each `run()` grabs a fresh frame by default. Pass `use_last=True` to reuse the last frame the camera grabbed (useful for running multiple detections on the same frame).

In [6]:
cnt = vc.detection("cnt1")
blob = vc.detection("blob1")

valid_cnt = cnt.run()
valid_blob = blob.run(use_last=True)      # reuses the frame cnt just grabbed

print("cnt detections: ", len(valid_cnt))
print("blob detections:", len(valid_blob))
for r in valid_cnt[:3]:
    print(" ", r)

cnt detections:  2
blob detections: 0
  {'timestamp': 1777055086.3869936, 'cls': 'cnt', 'conf': 1, 'center': [210, 308], 'corners': [[191, 306], [238, 292], [242, 306], [195, 319]], 'id': 11, 'ej': [0, 0, 0, 0, 0, 0, 0, 0], 'xyz': [-831.8443298339844, 268.0225074291229, 1708.5999250411987]}
  {'timestamp': 1777055086.3869936, 'cls': 'cnt', 'conf': 1, 'center': [439, 326], 'corners': [[0, 11], [847, 11], [847, 479], [0, 479]], 'id': 22, 'ej': [0, 0, 0, 0, 0, 0, 0, 0], 'xyz': [18.45932938158512, 71.86506688594818, 363.7000024318695]}


## 6. Fetch the annotated image and the binary threshold

Images come back as JPEG bytes over a binary WebSocket frame (fast, ~2 ms on localhost).

In [ ]:
from IPython.display import Image, display

jpeg_img, meta = cnt.get_img(type="img")
print("annotated:", meta)
display(Image(data=jpeg_img))

jpeg_thr, meta = cnt.get_img(type="img_thr")
print("threshold:", meta)
display(Image(data=jpeg_thr))

## 7. Read Detection attributes via the proxy

Attribute access works too — call the name with parens: `det.retval()`, `det.frame_mat_inv()`.
Big numpy arrays (images) come back as `{"_placeholder": "ndarray", "shape": ..., "dtype": ...}`. Use `.get_img(...)` to fetch actual pixels.

In [8]:
import pprint

retval = cnt.retval()
pprint.pp({
    "valid_count":   len(retval["valid"]),
    "all_count":     len(retval["all"]),
    "frame_mat_inv": retval["frame_mat_inv"],
    "camera_data_keys": list(retval["camera_data"].keys()),
    "camera_data.K": retval["camera_data"]["K"],
    "camera_data.img": retval["camera_data"]["img"],   # placeholder stub
})

{'valid_count': 2,
 'all_count': 41,
 'frame_mat_inv': [[1.0, 0.0, 0.0, 0.0],
                   [0.0, 1.0, 0.0, 0.0],
                   [0.0, 0.0, 1.0, 0.0],
                   [0.0, 0.0, 0.0, 1.0]],
 'camera_data_keys': ['depth_frame',
                      'ir_frame',
                      'color_frame',
                      'depth_img',
                      'ir_img',
                      'color_img',
                      'depth_int',
                      'frames',
                      'joint',
                      'timestamp',
                      'K',
                      'D',
                      'img',
                      'img_roi'],
 'camera_data.K': [[430.4017028808594, 0.0, 417.181640625],
                   [0.0, 429.86138916015625, 241.1890106201172],
                   [0.0, 0.0, 1.0]],
 'camera_data.img': {'_placeholder': 'ndarray',
                     'shape': [480, 848, 3],
                     'dtype': 'uint8',
                     'size': 1221120}}


## 8. Call any Camera method via the proxy

Anything safe on `camera.Camera` forwards as-is. New methods added upstream just work — no client change.

In [9]:
cam = vc.camera(serial_number)

print("exposure:", cam.get_exposure())
print("temp:",     cam.get_temp())

# turn off auto-exposure, pin a value, then restore
cam.auto_exposure(False)
cam.set_exposure(2000)
print("after set:", cam.get_exposure())
cam.auto_exposure(True)

exposure: 33000.0
temp: 34.0
after set: 2000.0


2000.0

## 9. Pixel <-> world coordinate conversion

In [10]:
print("xyz at pixel [424, 240]:", cnt.xyz([424, 240]))
print("pixel for xyz [0, 0, 500]:", cnt.pixel([0, 0, 500]))

xyz at pixel [424, 240]: [5.595286842435598, -0.9770586621016264, 353.1999886035919]
pixel for xyz [0, 0, 500]: [0, 0]


## 10. Grasp planning

`Detection.grasp(...)` wraps `collision_free_rvec` — returns a collision-free rvec for picking the target, given the current scene.
Skipped unless there are detections and you pass a valid `target_id`.

In [11]:
if valid_cnt:
    tid = valid_cnt[0]["id"]
    rvec = cnt.grasp(
        target_id=tid,
        target_rvec=[0, 0, 0],
        gripper_opening=30,
        finger_wdith=10,
        finger_location=[0, 180],
    )
    print("best rvec for target", tid, ":", rvec)
else:
    print("no detections — skip grasp")

best rvec for target 11 : [0.0, 0.0, 179.99999502168694]


## 11. (Optional) Robot control via the proxy

Any method on `dorna2.Dorna` works: `play`, `joint`, `recv`, kinematic forward/inverse, etc.

In [ ]:
# robot = vc.robot("192.168.1.50")
# print("joint:", robot.joint())
# print("recv:",  robot.recv())
# robot.play(msg={"cmd": "jmove", "rel": 1, "j0": 10, "vel": 50})

## 12. Error handling

The server returns structured errors; the client raises `VisionServerError` with a `code` and `msg`.

In [ ]:
from dorna_vision_client import VisionServerError

try:
    vc.detection_run("does_not_exist")
except VisionServerError as e:
    print("caught:", e.code, "-", e.msg)

## 13. Cleanup

`vc.close()` removes this client's detections but leaves shared cameras and robots on the server. Call `camera_remove` / `robot_remove` explicitly when you want the underlying resource to be released.

In [12]:
vc.detection_remove("cnt1")
vc.detection_remove("blob1")
vc.camera_remove(serial_number)
# vc.robot_remove("192.168.1.50")
vc.close()